---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-37: LangChain <b>Retrievers</b></h1>

# Learning agenda of this notebook  

1. Understanding LangChain Retrievers Components and Retriever Types in LangChain
2. Understanding `VectorStoreRetriever`
3. Hands-On Examples of using `VectorStoreRetriever` with Available Search Types in `as_retriever()`
    - Example 1: Standard Similarity Search (`similarity`)
    - Example 2: Similarity Score Threshold Search (`similarity_score_threshold`)
    - Example 3: Maximal Marginal Relevance Search (`mmr`)
4. Understanding `WikipediaRetriever`
    - Hands-On Examples of using `WikipediaRetriever`
5. Understanding `MultiQueryRetriever`
    - Hands-On Examples of using `MultiQueryRetriever`

# <span style='background :lightgreen' >Recap: Core Components of [LangChain](https://github.com/langchain-ai/langchain)</span>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain is an open source framework that provides us with a set of tools and abstractions that make it easier for us to create complex LLM-powered applications.</div></h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Langchain components can be chained together to create sophisticated AI applications like chatbots, question-answering systems, and intelligent agents.</div></h3>


| Component | Description | Key Benefit |
|-----------|--------------|-------------|
| **Models** | LangChain's "universal remote control" for AI—one interface to rule them all, freeing developers from vendor-specific complexity and enabling true AI provider independence. | Write once, use with any AI model |
| **Prompts** | Reusable, parameterized templates that standardize how you communicate with AI models. | Consistent messaging with easy updates | 
| **Memory** | LangChain Memory enables AI models to maintain context across multiple interactions in a conversation. Without memory, each interaction is independent - the model has no knowledge of previous exchanges. | Natural, flowing conversations |
| **Chains** | LangChain’s “AI Assembly Lines” — they connect multiple models, prompts, or tools together into a logical flow of reasoning, automating complex multi-step tasks with simplicity and consistency. | Transform complex tasks into simple sequences | 
| **Indexes** | Indexes are LangChain's smart knowledge management systems that convert your raw documents (PDFs, websites, databases) into searchable libraries where AI models can quickly find and retrieve only the most relevant information w/o exceeding context limits. | Fast retrieval, reduced API costs | 
| **Agents** | LangChain Agents are the “decision-making brains” of your AI system — they enable models to reason, plan, and dynamically decide which tools to use and what actions to take in order to achieve a user’s goal.| Autonomous problem-solving without manual programming |

# <span style='background :lightgreen' >Recap of LangChain Indexes Component</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain Indexes consist of four interconnected components that work together to enable Retrieval Augmented Generation (RAG).</div></h3>

<img align="right" width="800" src="../images/r3.png"  >

- [**Document Loaders**](https://docs.langchain.com/oss/javascript/integrations/document_loaders) provide a standard interface (160+ integrations) for reading data from different sources ike PDFs, web pages, Google Drive, Slack, Notion, and more into LangChain’s Document format. This ensures that data can be handled consistently regardless of the source. Common categories include:
    - **TextLoaders**  for plain .txt files (`CSVLoader`, `Docx2txtLoader`, `JSONLoader`, `PyPDFLoader`, `MarkdownLoader` etc)
    - **Web loaders** for ingesting data from online or cloud-based sources. (`WebBaseLoader`, `SitemapLoader`, `S3FileLoader`, `GCSFileLoader`, etc)
    - **Productivity tools** for pulling data from collaboration and enterprise platforms. (`GoogleDriveLoader`, `YouTubeLoader`, `TwitterLoader`, `WikipediaLoader`, `NotionPageLoader`, `ConfluenceLoader` etc)
- [**Text Splitters**](https://docs.langchain.com/oss/javascript/integrations/splitters/index#text-splitters) break large docs into smaller chunks that will be retrievable individually, ensuring that each chunk contains meaningful information while staying within model context window limit. Main strategies are:
    - **Text structure-based:** Splits hierarchically (paragraphs → sentences → words)
    - **Length-based:** Splits by token or character count
    - **Document structure-based:** Splits based on format (HTML, Markdown, code)
- [**Vector Stores**](https://docs.langchain.com/oss/python/integrations/vectorstores/index) are specialized databases for storing and searching embeddings. They store embedded documents and perform similarity search to find semantically similar content. LangChain provides a unified interface with methods like addDocuments, delete, and similaritySearch. Popular options include Chroma, Pinecone, Qdrant, FAISS, MongoDB Atlas, and in-memory stores for testing.
- [**Retrievers**](https://docs.langchain.com/oss/python/integrations/retrievers/index#retrievers) accepts an unstructured query as input, search through the vector database to find the most relevant document chunks and return a list of Documents as output.

# <span style='background :lightgreen' >1. Understanding LangChain Retrievers Component</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Retrievers in LangChain accepts an unstructured query as input, search through the vector database to find the most relevant document chunks and return a list of Documents as output. </div></h3>

<img align="right" width="800" src="../images/RAGf.png"  >

- 🔍 **Retriever = Information Bridge:** A retriever acts as the bridge between the user’s query and the stored knowledge base, identifying the most relevant pieces of information to support the LLM’s response.
- 🧠 **Core Idea — Relevance Retrieval:** Instead of generating answers directly, retrievers *fetch contextually similar documents* (via semantic or keyword matching) that the LLM can use to ground its reasoning.
- 🧩 **Embedding or Keyword Driven:** Retrievers can use **vector embeddings** for semantic similarity or **keyword-based scoring** (like BM25) for exact term matches, depending on the application.
- ⚙️ **Composable and Interchangeable:** LangChain retrievers follow a standard interface (`get_relevant_documents()`) so different retrievers (vector, BM25, hybrid, etc.) can be swapped or combined seamlessly.
- 🧮 **Supports Hybrid and Enhanced Retrieval:** Advanced retrievers (e.g., Ensemble, ContextualCompression, SelfQuery) can combine multiple retrieval methods or leverage an LLM to refine, summarize, or interpret retrieved data.
- 🚀 **Foundation for RAG Pipelines:** In Retrieval-Augmented Generation (RAG), retrievers provide the *context window* for the LLM, ensuring responses are factually grounded and domain-specific.

## a. Common Retriever Types in LangChain

| **Category**                         | **Example Classes**                                                | **Uses Embeddings?** | **Description / What It’s Used For**                                                                                        |
| ------------------------------------ | ------------------------------------------------------------------ | -------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| **Vector-Based Retriever**           | `VectorStoreRetriever`                                             | ✅                    | Retrieves documents using semantic similarity of embeddings stored in a vector database (e.g., Chroma, FAISS, Pinecone).    |                     |
| **Multi-Vector Retriever**           | `MultiVectorRetriever`                                             | ✅                    | Stores multiple embeddings per document (e.g., title + body + summary) for multimodal or fine-grained retrieval.            |
| **Parent Document Retriever**        | `ParentDocumentRetriever`                                          | ✅                    | Retrieves entire parent documents after chunk-level similarity search — preserves full context.                             |
| **Contextual Compression Retriever** | `ContextualCompressionRetriever`                                   | ✅                    | Wraps another retriever and uses an LLM to compress or summarize retrieved text before passing to the model.                |
| **Time-Weighted Retriever**          | `TimeWeightedVectorRetriever`                                      | ✅                    | Prioritizes recent documents while still considering semantic similarity — ideal for time-sensitive data (e.g., chat logs). |
| **Self-Query Retriever**             | `SelfQueryRetriever`                                               | ✅                    | Uses an LLM to interpret user intent and convert natural language queries into structured filters + embedding search.       |
| **Web/API Retriever**                | `TavilySearchAPIRetriever`, `WikipediaRetriever`, `ArxivRetriever` | ❌                    | Retrieves up-to-date information from online sources or APIs instead of local documents.                                    |
| **Memory-Based Retriever**           | `ConversationBufferMemory`, `VectorStoreRetrieverMemory`           | ✅ / ❌                | Used in chatbots to recall past interactions or relevant context from memory.  
| **Keyword-Based Retriever**          | `BM25Retriever`, `TFIDFRetriever`                                  | ❌                    | Performs lexical search using keyword matching; useful when exact term matches are important.                               |
| **Hybrid / Ensemble Retriever**      | `EnsembleRetriever`                                                | ✅ / ❌                | Combines multiple retrievers (e.g., vector + keyword) using weights or rank fusion to improve accuracy.|


>- **Re-ranking is a fascinating piece of the RAG puzzle because it doesn't replace any retriever rather it sits *between* the retriever and the LLM, acting as a second-pass quality filter.** (More on this in next Notebook)

# <span style='background :lightgreen' >2. Understanding `VectorStoreRetriever`</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">A VectorStoreRetriever in LangChain is the most common type of retriever that lets you search and fetch documents from a vector store based on semantic similarity using vector embeddings. </div></h3>


- The `as_retriever()` method converts a vector store (like Chroma, FAISS, etc.) into a VectorStoreRetriever object — a standardized wrapper that exposes standardized methods like `.invoke(query)`,  `.get_relevant_documents(query)` and `.aget_relevant_documents(query)`


| Important Arguments            | Type   | Description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| ------------------- | ------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **`search_type`**   | `str`  | Defines the retrieval algorithm used to fetch documents. Common values:  <br><br> 🔹 **`"similarity"`** – *Pure relevance*: standard cosine similarity search. Ideal for factual Q&A and general retrieval. <br>  • ✅ *Fast, simple, highly relevant* <br>  • ⚠️ *May return redundant results* <br><br> 🔹 **`"similarity_score_threshold"`** – *Quality control*: returns only documents above a given similarity score threshold. Best for high-confidence retrievals. <br>  • ✅ *Filters low-quality matches* <br>  • ⚠️ *May return few/no results* <br><br> 🔹 **`"mmr"`** – *Relevance + Diversity*: Maximal Marginal Relevance, balancing similarity with diversity. Ideal for research or exploratory retrieval. <br>  • ✅ *Diverse results, avoids redundancy* <br>  • ⚠️ *Slightly slower* |
| **`search_kwargs`** | `dict` | Dictionary of parameters passed to the search method. Examples:  <br>• `{"k": 5}` → number of results to return. <br>• `{"k": 5, "fetch_k": 20, "lambda_mult": 0.5}` → for MMR. <br>• `{"score_threshold": 0.8}` → for `similarity_score_threshold`.                                                                                                                                                        

- **Quick Decision Tree for the use of `search_type` argument:**
```
Do you need diverse results?
│
├─ NO → Use search_type="similarity"
│        • Simple, fast, and returns top-k most relevant results
│        • May include redundant or overly similar documents
│
├─ NO, but want only high-confidence matches?
│        → Use search_type="similarity_score_threshold"
│          • Filters out low-similarity results using a score cutoff
│          • Ensures only strong, relevant matches are returned
│          • Example: search_kwargs={"score_threshold": 0.7, "k": 5}
│
└─ YES → Use search_type="mmr"
         • Balances relevance with diversity to reduce redundancy
         │
         ├─ Want balanced results? 
         │    → lambda_mult = 0.5
         │
         ├─ Need high relevance, avoid near-duplicates?
         │    → lambda_mult = 0.7–0.9
         │
         └─ Want maximum diversity?
              → lambda_mult = 0.2–0.4
```



# <span style='background :lightgreen' >3. Hands-On Examples of using `VectorStoreRetriever` with Available Search Types in `as_retriever()`</span>
### Setup your Vector Store
```



┌─────────────────┐
│   Document      │ Convert all your documents into vectors using  an embedding model
│   Embedding     │ [0.2, 0.8, 0.1, ...]
└─────────┬───────┘
          │
          ▼
┌─────────────────┐
│   Store         │ Store the embeddings of your documents in a vector store (like FAISS, Chroma, Weaviate).
│   Embedding     │ 
└─────────┬───────┘
          │
          ▼
```



### How VectorStoreRetriever Works?:
```
┌─────────────────┐
│   User Query    │ "What is the capital of Pakistan?"
└─────────┬───────┘
          │  User input (natural language)
          ▼
┌─────────────────┐
│   Query         │ Convert text to embeddings
│   Embedding     │ [0.2, 0.8, 0.1, ...]
└─────────┬───────┘
          │  Uses the embedding model defined in vector store
          ▼
┌─────────────────┐
│   Similarity    │ Compare query vector with all document embeddings
│   Calculation   │ Cosine / Euclidean / Inner Product distance
└─────────┬───────┘
          │
          ▼
┌────────────────────────────────────────────────────────────────┐
│   Document Ranking (based on `search_type`)                    │
│                                                                │
│   • search_type="similarity"                                   │
│       → Ranks all documents purely by similarity score         │
│                                                                │
│   • search_type="similarity_score_threshold"                   │
│       → Filters out documents below a minimum score threshold  │
│         (e.g., score < 0.7 ignored)                            │
│                                                                │
│   • search_type="mmr"                                          │
│       → Applies *Maximal Marginal Relevance* to balance        │
│         relevance and diversity (reduces redundancy)           │
└─────────┬──────────────────────────────────────────────────────┘
          │
          ▼
┌─────────────────┐
│   Top-k         │ Selects top-k results (default: 3–5)         
│   Selection     │ `similarity_score_threshold` may reduce count 
└─────────┬───────┘
          │
          ▼
┌─────────────────┐
│   Context       │ Formats selected documents for the LLM input  
│   Delivery      │ Ensures total context fits model’s token limit 
└─────────────────┘
```

## a. Load Documents Embeddings from your Database/Datastore

In [1]:
# ------------------------------------------------------------------------------------------------------------------------------------------------------
#  Option 1: Load the data from the existing Chroma database that we have created above inside "./RAG-data/chroma_db/")
# -------------------------------------------------------------------------------------------------------------------------------------------------------
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_chroma import Chroma

local_embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")      #"all-mpnet-base-v2"
vectordb = Chroma(
    collection_name="rag_collection",           # CRITICAL: Must match the name you used, while creating the db
    embedding_function=local_embedder,          # CRITICAL: Must be the same model used while creating the db
    persist_directory="./RAG-data/chroma_db/",  # This is your Chroma vector database object containing embedded documents
    collection_metadata={"hnsw:space": "cosine"}  
)
print(f"Chroma database loaded successfully")
print(f"   Total documents in database: {vectordb._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma database loaded successfully
   Total documents in database: 14


## b. Example 1: Standard Similarity Search (`similarity`)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Similarity search retrieves the most relevant documents by ranking them based on their cosine similarity to the query, focusing purely on relevance without considering redundancy.</div></h3>

- **How it Works**
    - Calculates similarity score between query and each document
    - Returns top-k documents with highest similarity scores
    - Problem: May return very similar or duplicate documents
- **Example Configuration**
```python
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
```

In [2]:
query = "What is the capital of Pakistan?"

# Convert vector store to retriever
retriever = vectordb.as_retriever(
    search_type="similarity",           # Type of search to perform: 'similarity', 'mmr', etc.
    search_kwargs={"k": 4},             # Dictionary of search parameters, default is none
    tags=None,             # List of tags for tracing/monitoring
    metadata=None,         # Dictionary of metadata
    verbose=False          # Enable verbose logging
)


print(f"   Retriever type: {type(retriever)}")

results = retriever.invoke(query)     # The retriever.invokesimilarity_search() returns list containing Document objects only

print(f"\n🔍 Query: '{query}'")
# Display results
print(f"Found {len(results)} documents:\n")
for i, doc in enumerate(results, 1):
    print(f"   Content {i}: {doc.page_content[:200]}...")
    print(f"   Metadata{i}: {doc.metadata}\n")

   Retriever type: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>

🔍 Query: 'What is the capital of Pakistan?'
Found 4 documents:

   Content 1: Islamabad is the capital of Pakistan...
   Metadata1: {'source': 'manual_entry', 'country': 'Pakistan', 'type': 'geography'}

   Content 2: Rome is the capital of Italy...
   Metadata2: {'source': 'manual_entry', 'country': 'Italy', 'type': 'geography'}

   Content 3: Paris is the capital of France...
   Metadata3: {'source': 'manual_entry', 'type': 'geography', 'country': 'France'}

   Content 4: Rising sea levels due to climate change threaten coastal cities like Mumbai and New York....
   Metadata4: {'category': 'Environment', 'source': 'news_report', 'topic': 'Climate Change', 'region': 'Global'}



## b. Example 2: Similarity Score Threshold Search (`similarity_score_threshold`)

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"> Similarity Score Threshold search filters documents based on a minimum similarity score, ensuring that only highly relevant and high-confidence matches are retrieved.</div></h3>
    
- **How it Works**
    - Computes similarity scores between the query and all documents
    - Returns only those documents whose scores exceed a specified threshold
    - Useful when you want high-confidence matches and to filter out weakly related results
    - Trade-off: May return fewer or even zero documents if the threshold is too high
- **Example Configuration**
```python
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.75,  # minimum similarity score to include
        "k": 4                    # max number of documents to return
    }
)
```


In [3]:
query = "What is the capital of Pakistan?"

# Convert vector store to retriever
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold", # Use score threshold–based retrieval
    search_kwargs={
        "score_threshold": 0.3,               # Only keeps documents whose similarity score ≥ threshold (range: 0–1)
        "k": 4                                # Optional: maximum number of results to return
    }
)

print(f"Retriever type: {type(retriever)}")

# Perform retrieval
results = retriever.invoke(query)     # Returns a list of Document objects only. To get the score use `retriever.vectorstore.similarity_search_with_score()`.

print(f"\n🔍 Query: '{query}'")
print(f"Found {len(results)} documents:\n")
for i, doc in enumerate(results, 1):
    print(f"   Content {i}: {doc.page_content[:200]}...")


Retriever type: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>

🔍 Query: 'What is the capital of Pakistan?'
Found 2 documents:

   Content 1: Islamabad is the capital of Pakistan...
   Content 2: Rome is the capital of Italy...


## c. Example 3: Maximal Marginal Relevance Search (`mmr`)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">How can we pick results that are not only relevant to the query but also different from eachother.. </div></h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query. </div></h3>

- **How it Works**
    - Balances **relevance** (similarity to the query) with **diversity** (difference among returned documents)
    - Ensures that the retrieved documents are both **useful** and **non-redundant**
    - Particularly effective for **research**, **summarization**, or **broad topic exploration**
    - Slightly slower and may include documents with lower direct similarity but higher information diversity

- **Example Configuration**
```python
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,            # number of final results to return
        "fetch_k": 20,     # number of candidates to consider before applying MMR
        "lambda_mult": 0.5 # balance between relevance (1.0) and diversity (0.0)
    }
)
```
- **The `lambda_mult` parameter is crucial for controlling the relevance-diversity trade-off in MMR.**

```
┌─────────────────────────────────────────────────────────────────┐
│                    lambda_mult Spectrum                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  0.0                    0.5                    1.0              │
│   │                      │                      │               │
│   └──────────────────────┴──────────────────────┘               │
│   Pure                 Balanced              Pure               │
│   Diversity                                Relevance            │
│                                                                 │
│   ↓                      ↓                      ↓               │
│   Max variety         Best of both         Most similar         │
│   May lose focus      (RECOMMENDED)         May be redundant    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

- **Understanding `fetch_k` Parameter**

```
┌──────────────────────────────────────────────────────────────────┐
│                    How MMR Works Internally                      │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  Step 1: Fetch fetch_k candidates (similarity search)            │
│  ┌─────────────────────────────────────────────────┐             │
│  │  Fetch top 20 most similar documents            │             │
│  │  [D1, D2, D3, D4, D5, D6, D7, ... D20]          │             │
│  └─────────────────────────────────────────────────┘             │
│                         ↓                                        │
│  Step 2: Apply MMR algorithm to select k diverse docs            │
│  ┌─────────────────────────────────────────────────┐             │
│  │  Select 5 that are relevant AND diverse         │             │
│  │  [D1, D4, D8, D12, D18]                         │             │
│  └─────────────────────────────────────────────────┘             │
│                         ↓                                        │
│  Step 3: Return k documents                                      │
│  ┌─────────────────────────────────────────────────┐             │
│  │  Final results: 5 diverse documents             │             │
│  └─────────────────────────────────────────────────┘             │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

**Rule of Thumb**: Set `fetch_k` to be **2-5 times larger than k**
- `k=3` → `fetch_k=10-15`
- `k=5` → `fetch_k=20-25`
- `k=10` → `fetch_k=30-50`


In [4]:
query = "How is climate change affecting different regions?"
#query = "Tell me about LangChain and related AI tools"
#query = "What is LangChain used for?"
#query = "What is the capital of Pakistan?"
# Convert vector store to retriever using MMR for diversity
retriever = vectordb.as_retriever(
    search_type="mmr",                     # Use Maximal Marginal Relevance (MMR)
    search_kwargs={
        "k": 4,                            # Number of diverse results to return
        "fetch_k": 20,                     # Number of initial candidates to consider
        "lambda_mult": 0.5                 # Balance between relevance (1.0) and diversity (0.0)
    }
)

print(f"Retriever type: {type(retriever)}")

# Perform retrieval
results = retriever.invoke(query)  # Returns a diverse set of Document objects

print(f"\n🔍 Query: '{query}'")
print(f"Found {len(results)} diverse documents:\n")

for i, doc in enumerate(results, 1):
    print(f"   Content {i}: {doc.page_content[:200]}...")


Retriever type: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>

🔍 Query: 'How is climate change affecting different regions?'
Found 4 diverse documents:

   Content 1: Climate change is causing glaciers to melt rapidly in the Arctic region....
   Content 2: Rising sea levels due to climate change threaten coastal cities like Mumbai and New York....
   Content 3: LangChain is used to build LLM based applications....
   Content 4: Paris is the capital of France...


# <span style='background :lightgreen' >4. Hands-On Examples of using `WikipediaRetriever`</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">A WikipediaRetriever in LangChain is a retriever that queries the Wikipedia API to fetch relevant content for a given query. </div></h3>

- **How it works?**
    - You give it a query.
    - It sends the query to Wikipedia API
    - It retrieves the most relevant articles
    - It returns them as LangChain Document objects 
- To install this retriever on your system, run the following command:
```
uv add wikipedia
```

##  Hands-On Examples of using `WikipediaRetriever` 

In [5]:
from langchain_community.retrievers import WikipediaRetriever

# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

# Define your query
query = "Durand Line agreement"
# Get relevant Wikipedia documents
docs = retriever.invoke(query)
print(docs)

[Document(metadata={'title': 'Durand Line', 'summary': 'The Durand Line, also known as the Afghanistan–Pakistan border, is a 2,640-kilometre (1,640 mi) international border between Afghanistan and Pakistan. The western end runs to the border with Iran and the eastern end to the border with China.\nThe Durand Line was established in 1893 as the international border between the Emirate of Afghanistan and the British Indian Empire by Mortimer Durand, a British diplomat of the Indian Civil Service, and Abdur Rahman Khan, the Emir of Afghanistan, to fix the limit of their respective spheres of influence and improve diplomatic relations and trade. Britain considered Afghanistan to be an independent state at the time, although they controlled its foreign affairs and diplomatic relations.\nThe single-page Agreement, dated 12 November 1893, contains seven short articles, including a commitment not to exercise interference beyond the Durand Line. A joint British-Afghan demarcation survey took pl

In [6]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

# Define your query
query = "Durand Line agreement"
# Get relevant Wikipedia documents
docs = retriever.invoke(query)

print(f"\033[1m{'='*80}\033[0m")
print(f"\033[1mRETRIEVAL SUMMARY\033[0m")
print(f"\033[1m{'='*80}\033[0m")
print(f"Query: '{query}'")
print(f"Total documents retrieved: {len(docs)}")
print(f"Return type: {type(docs)}")
print(f"Document type: {type(docs[0]) if docs else 'N/A'}\n")

RETRIEVAL SUMMARY
Query: 'Durand Line agreement'
Total documents retrieved: 2
Return type: <class 'list'>
Document type: <class 'langchain_core.documents.base.Document'>



In [7]:
# Detailed inspection of each document
for i, doc in enumerate(docs):
    print(f"\033[1m{'─'*80}\033[0m")
    print(f"\033[1m📄 DOCUMENT {i+1}\033[0m")
    print(f"\033[1m{'─'*80}\033[0m")
    
    # Metadata
    print(f"\n\033[1m🏷️  METADATA:\033[0m")
    for key, value in doc.metadata.items():
        print(f"  • {key}: {value}")
    
    # Content summary
    print(f"\n\033[1m📝 CONTENT:\033[0m")
    print(f"  • Length: {len(doc.page_content)} characters")
    print(f"  • Word count: ~{len(doc.page_content.split())} words")
    print(f"  • Lines: {len(doc.page_content.splitlines())}")
    
    # First 500 characters
    print(f"\n\033[1m📖 PREVIEW (first 500 chars):\033[0m")
    print(f"  {doc.page_content[:500]}...")
    
    print()  # blank line between documents

────────────────────────────────────────────────────────────────────────────────
📄 DOCUMENT 1
────────────────────────────────────────────────────────────────────────────────

🏷️  METADATA:
  • title: Durand Line
  • summary: The Durand Line, also known as the Afghanistan–Pakistan border, is a 2,640-kilometre (1,640 mi) international border between Afghanistan and Pakistan. The western end runs to the border with Iran and the eastern end to the border with China.
The Durand Line was established in 1893 as the international border between the Emirate of Afghanistan and the British Indian Empire by Mortimer Durand, a British diplomat of the Indian Civil Service, and Abdur Rahman Khan, the Emir of Afghanistan, to fix the limit of their respective spheres of influence and improve diplomatic relations and trade. Britain considered Afghanistan to be an independent state at the time, although they controlled its foreign affairs and diplomatic relations.
The single-page Agreement, dated 12 Nov

# <span style='background :lightgreen' >5. Hands-On Examples of using `MultiQueryRetriever`</span>

<h3 align="center"> <div class="alert alert-success" color=magenta style="margin: 20px"> The <b>MultiQueryRetriever</b> enhances retrieval quality by generating several semantically different variations of your original query using an LLM. It then searches the vector store with each version, combining the results to improve recall and capture more diverse, relevant documents. </div> </h3>

- **Key Idea:**
    - Instead of relying on one phrasing of a question, it uses multiple reformulations.
    - This helps overcome limitations of wording differences between user queries and stored documents.
    - Especially useful for complex, ambiguous, or broad information-seeking tasks.

- **How it Works?**
    - Takes your original query (e.g., “What is LangChain used for?”)
    - Uses an LLM (like GPT-3.5 or GPT-4) to generate multiple semantically different variants, such as:
        - “How is LangChain applied in LLM pipelines?”
        - “What are the main use cases of LangChain?”
        - “Why do developers use LangChain for AI apps?”
- Performs retrieval for each sub-query across the same vector store.

```mermaid
graph TD
    O["🔍 ORIGINAL QUERY<br/>'What is LangChain used for?'"]
    LLM["🧠 LLM GENERATOR<br/>(GPT-3.5 / GPT-4)<br/>Creates Multiple Variations"]

    Q1["📄 SUB-QUERY 1<br/>'How is LangChain applied in LLM pipelines?'"]
    Q2["📄 SUB-QUERY 2<br/>'What are the main use cases of LangChain?'"]
    Q3["📄 SUB-QUERY 3<br/>'Why do developers use LangChain for AI apps?'"]

    S1["🔎 VECTOR SEARCH<br/>Database Query"]
    S2["🔎 VECTOR SEARCH<br/>Database Query"]
    S3["🔎 VECTOR SEARCH<br/>Database Query"]

    R1["📚 RESULT SET 1<br/>LLM Integration Docs"]
    R2["📚 RESULT SET 2<br/>Use Case Articles"]
    R3["📚 RESULT SET 3<br/>Developer Guides"]

    COMB["🔗 COMBINE & DEDUPLICATE<br/>Merge All Results<br/>Remove Duplicates"]
    FINAL["🏁 FINAL COMPREHENSIVE RESULTS<br/><b>LangChain Overview</b><br/>• LLM integration resources<br/>• Application examples<br/>• Developer documentation"]

    %% connections
    O --> LLM
    LLM --> Q1
    LLM --> Q2
    LLM --> Q3

    Q1 --> S1 --> R1 --> COMB
    Q2 --> S2 --> R2 --> COMB
    Q3 --> S3 --> R3 --> COMB

    COMB --> FINAL

    %% styling (Jupyter-friendly)
    style O fill:#a7c7e7,stroke:#1f618d,stroke-width:2px
    style LLM fill:#f5b971,stroke:#d35400,stroke-width:2px
    style Q1,Q2,Q3 fill:#79b8b8,stroke:#0b5345,stroke-width:1px
    style S1,S2,S3 fill:#f77b42,stroke:#b03a2e,stroke-width:1px
    style R1,R2,R3 fill:#9fd68f,stroke:#196f3d,stroke-width:1px
    style COMB fill:#b181d6,stroke:#6c3483,stroke-width:2px
    style FINAL fill:#57c36d,stroke:#145a32,stroke-width:2px
```


##  Hands-On Example of using `MultiQueryRetriever` 

In [8]:
# ---------------------------------------------
# Creating  a new In-Memory Chroma Vector Store
# ---------------------------------------------
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_chroma import Chroma

docs = [
    Document(page_content="Python is widely used for building data science and machine learning applications."),
    Document(page_content="LangChain helps developers connect LLMs with external tools, APIs, and databases."),
    Document(page_content="The capital of Pakistan is Islamabad."),
    Document(page_content="Karachi is the largest city and financial hub of Pakistan."),
    Document(page_content="Developers use LangChain to build retrieval-augmented generation (RAG) systems."),
    Document(page_content="Machine learning models can be trained using TensorFlow or PyTorch."),
    Document(page_content="RAG systems improve LLM accuracy by fetching relevant documents from vector databases.")
    ]
local_embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Chroma class is a vector store wrapper in LangChain for ChromaDB. It provides a high-level Python interface for storing document embeddings, performing similarity searches and integrating with LLM pipelines
vectorstore = Chroma(
    collection_name = "rag_collection",      # Name of the collection inside Chroma
    embedding_function = local_embedder,     # The embedding model used to generate vector representations
    collection_metadata={"hnsw:space": "cosine"}  # For Euclidean (L2) distance: "l2", For Inner product: "ip"
)

#vectorstore.delete_collection()  # If the collection already exist then use this
vectorstore.reset_collection() # This will allow you to repeatedly execute this cell code and it will not add the documents again and again
# Add the all the documents  to the In Memory Chroma db using its `add_documents()` method that computes embeddings for each document and stores them
vectorstore.add_documents(documents = docs)

data = vectorstore.get(include=["documents", "metadatas", "embeddings"]) # returns a dictionary containing 'documents', 'metadatas', and  `embeddings`

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
# --------------------------
# Using VectorStoreRetriever
# --------------------------
query = "How do developers connect LLMs to other systems or databases?"

retriever_simple = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
results_simple = retriever_simple.invoke(query)

print("🔹 Simple Retriever Results:\n")
for i, doc in enumerate(results_simple, 1):
    print(f"{i}. {doc.page_content}\n")


🔹 Simple Retriever Results:

1. LangChain helps developers connect LLMs with external tools, APIs, and databases.

2. RAG systems improve LLM accuracy by fetching relevant documents from vector databases.

3. Machine learning models can be trained using TensorFlow or PyTorch.



In [10]:
# --------------------------
# Using MultiQueryRetriever
# --------------------------
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_classic.retrievers import MultiQueryRetriever

load_dotenv('../keys/.env', override=True) 
groq_api_key = os.getenv('GROQ_API_KEY')


llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

retriever_multi = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
    llm=llm
)

results_multi = retriever_multi.invoke(query)

print("🔹 MultiQueryRetriever Results:\n")
for i, doc in enumerate(results_multi, 1):
    print(f"{i}. {doc.page_content}\n")


🔹 MultiQueryRetriever Results:

1. LangChain helps developers connect LLMs with external tools, APIs, and databases.

2. Machine learning models can be trained using TensorFlow or PyTorch.

3. Developers use LangChain to build retrieval-augmented generation (RAG) systems.

4. RAG systems improve LLM accuracy by fetching relevant documents from vector databases.



>- Performs independent similarity searches for each rewritten query, each returning top k results.
>- Combines all results into a single unique set of Documents (using metadata and content deduplication). So the final result can be more than k

# <span style='background :lightgreen' >6. To DO:</span>

```python
from langchain_classic.retrievers import ContextualCompressionRetriever, ParentDocumentRetriever, TimeWeightedVectorStoreRetriever, EnsembleRetriever, MultiVectorRetriever, SelfQueryRetriever, BM25Retriever
```
- **`ContextualCompressionRetriever`** – Compresses or filters retrieved documents using an LLM to keep only the most relevant content.
- **`ParentDocumentRetriever`** – Retrieves smaller child chunks but returns their larger parent documents for richer context.
- **`TimeWeightedVectorStoreRetriever`** – Prioritizes more recent documents by combining recency with semantic similarity.
- **`EnsembleRetriever`** – Combines results from multiple retrievers (e.g., vector + keyword) for better overall recall and relevance.
- **`MultiVectorRetriever`** – Allows storing and retrieving multiple embeddings per document (e.g., for passages, summaries, or tables).
- **`SelfQueryRetriever`** – Uses an LLM to transform natural language queries into structured filter + search expressions.
- **`BM25Retriever`** – A classic keyword-based retriever using the BM25 ranking algorithm for lexical (non-embedding) search.
